tool construct: answering questions about "Huawei's latest phone" --> search tool

In [25]:
import os
from dotenv import load_dotenv

load_dotenv()


True

In [33]:
# MY VERSION

from serpapi import SerpApiClient

def search(query: str) -> str:
    try:
        api_key = os.getenv("SERPAPI_API_KEY")

        params = {
            "engine" : "google",
            "q" : query,
            "api_key" : api_key,
            "gl": "de",  
            "hl": "en", 
        }
        client = SerpApiClient(params)

        result = client.get_dict()


        # print(result.keys())
        # print(result.keys())
        # print(type(result["organic_results"]))
        # print(result["organic_results"][0].keys())
        # print(result["organic_results"][0]["title"])
        if "answer_box_list" in result:
            return "\n".join(result["answer_box_list"])

        if "answer_box" in result and "answer" in result["answer_box"]:
            return result["answer_box"]["answer"]

        if "knowledge_graph" in result:
            return result["knowledge_graph"]["description"]

        if "organic_results" in result:
            snippets = []
            for res in result["organic_results"][:3]:
                title = res.get("title", "")
                snippet = res.get("snippet", "")
                snippets.append(title + "/n" + snippet)

            return "\n\n".join(snippets)

        return "dont find out"
    
    except Exception as e:
        return f"Error occur when searching."




            



   



In [35]:
search("where is kangaroo from?")

"Kangaroo/nKangaroos are indigenous to Australia and New Guinea. tropical rainforests of New Guinea, far northeastern Queensland,\n\nKangaroos (Facts & Photos)/nThe word kangaroo derives from 'Gangurru', the name given to Eastern Grey Kangaroos by the Guuga Yimithirr people of Far North Queensland.(1)\n\nKangaroos - Wildlife/nKangaroos are some of Australia's most recognisable and well known native animals. They form an integral part of our natural ecosystems, playing ..."

In [ ]:
#Reference version
from serpapi import SerpApiClient

def search(query: str) -> str:
    """
    A practical web search engine tool based on SerpApi.
    It intelligently parses search results, prioritizing direct answers or knowledge graph information.
    """
    print(f"🔍 Executing [SerpApi] web search: {query}")
    try:
        api_key = os.getenv("SERPAPI_API_KEY")
        if not api_key:
            return "Error: SERPAPI_API_KEY not configured in .env file."

        params = {
            "engine": "google",
            "q": query,
            "api_key": api_key,
            "gl": "cn",  # Country code
            "hl": "zh-cn", # Language code
        }
        
        client = SerpApiClient(params)
        results = client.get_dict()
        
        # Intelligent parsing: prioritize finding the most direct answer
        if "answer_box_list" in results:
            return "\n".join(results["answer_box_list"])
        if "answer_box" in results and "answer" in results["answer_box"]:
            return results["answer_box"]["answer"]
        if "knowledge_graph" in results and "description" in results["knowledge_graph"]:
            return results["knowledge_graph"]["description"]
        if "organic_results" in results and results["organic_results"]:
            # If no direct answer, return summaries of the first three organic results
            snippets = [
                f"[{i+1}] {res.get('title', '')}\n{res.get('snippet', '')}"
                for i, res in enumerate(results["organic_results"][:3])
            ]
            return "\n\n".join(snippets)
        
        return f"Sorry, no information found about '{query}'."

    except Exception as e:
        return f"Error occurred during search: {e}"

In [44]:
# MY VERSION
from typing import Dict, Any
class ToolExecuter:
    def __init__(self):
        self.tools = {}

    def tool_register(self, name, description,func):
        if name not in self.tools:
            self.tools[name] = {"description": description, "func":func}

    def getTool(self,name):
        return self.tools.get(name, {}).get("func")
        
    

In [45]:
#test
def add(a,b):
    return a+b

executor = ToolExecuter()

executor.tool_register("add", 
                       "add 2 numbers",
                       add)

test_func = executor.getTool("add")

print(test_func(1,2))

executor.tool_register("search",
                       "A web search engine. Use this tool when you need to answer questions about current events, facts, and information not found in your knowledge base.",
                       search)

test_func2 = executor.getTool("search")

print(test_func2("who is the current president of the US?"))



3
President of the United States/nDonald Trump is the 47th and current president since January 20, 2025.

President Donald J. Trump/nAfter a landslide election victory in 2024, President Donald J. Trump is returning to the White House. President of the United States

Presidents, vice presidents, and first ladies/nThe 47th and current president of the United States is Donald John Trump. He was sworn into office on January 20, 2025.


In [ ]:
# Reference version










from typing import Dict, Any

class ToolExecutor:
    """
    A tool executor responsible for managing and executing tools.
    """
    def __init__(self):
        self.tools: Dict[str, Dict[str, Any]] = {}

    def registerTool(self, name: str, description: str, func: callable):
        """
        Register a new tool in the toolbox.
        """
        if name in self.tools:
            print(f"Warning: Tool '{name}' already exists and will be overwritten.")
        self.tools[name] = {"description": description, "func": func}
        print(f"Tool '{name}' registered.")

    def getTool(self, name: str) -> callable:
        """
        Get a tool's execution function by name.
        """
        return self.tools.get(name, {}).get("func")

    def getAvailableTools(self) -> str:
        """
        Get a formatted description string of all available tools.
        """
        return "\n".join([
            f"- {name}: {info['description']}" 
            for name, info in self.tools.items()
        ])
